<h2>Data Manipulation with Pandas</h2>

Pandas is a package built on top of NumPy, and provides an efficient implementation of a `DataFrame`. `DataFrames` are essentially multidimensional arrays with attached row and column labels, and often with heterogenous types and/or missing data. As well as offering a convenient storage interface for labeled data, Pandas implements a number of powerful data operations familiar to users of both database frameworks and spreadsheet programs.

NumPy's `ndarray` data structure provides essential features for the type of clean, well-organized data typically seen in numerical computing tasks. While it serves this purpose very well, its limitations become clear when we need more flexibility (attaching labels to data, working with missing data, etc.) and when attempting operations that do not map well to element-wise broadcasting (groupings, pivots, etc.), each of which is an important piece of analyzing the less structured data available in many forms in the world around us.

We'll explore the mechanics of using `Series`, `DataFrame`, and related structures effectively. These examples are from real datasets where appropriate.

In [1]:
import pandas

pandas.__version__

'1.2.4'

In [3]:
import pandas as pd

# Recall that you can easily access documentation with a question mark, or pd.<TAB>
pd?

Type:        module
String form: <module 'pandas' from '/Users/mike/opt/anaconda3/lib/python3.8/site-packages/pandas/__init__.py'>
File:        ~/opt/anaconda3/lib/python3.8/site-packages/pandas/__init__.py
Docstring:  
pandas - a powerful data analysis and manipulation library for Python

**pandas** is a Python package providing fast, flexible, and expressive data
structures designed to make working with "relational" or "labeled" data both
easy and intuitive. It aims to be the fundamental high-level building block for
doing practical, **real world** data analysis in Python. Additionally, it has
the broader goal of becoming **the most powerful and flexible open source data
analysis / manipulation tool available in any language**. It is already well on
its way toward this goal.

Main Features
-------------
Here are just a few of the things that pandas does well:

  - Easy handling of missing data in floating point as well as non-floating
    point data.
  - Size mutability: columns can 

In [5]:
import numpy as np

data = pd.Series([0.25, 0.5, 0.75, 1.0])
data

0    0.25
1    0.50
2    0.75
3    1.00
dtype: float64

The `Series` wraps both a sequence of values and a sequence of indices, which we can access with the `values` and `index` attributes. The `values` are simply a familiar NumPy array.

In [6]:
data.values

array([0.25, 0.5 , 0.75, 1.  ])

In [7]:
data.index

RangeIndex(start=0, stop=4, step=1)

In [8]:
type(data.values)

numpy.ndarray

Like with a NumPy array, data can be accessed by the associated index via the familiar Python square-bracket notation:

In [9]:
data[1]

0.5

In [10]:
data[1:3]

1    0.50
2    0.75
dtype: float64

The Pandas `Series` is much more general and flexible than the one-dimensional NumPy array that it emulates.

From what we've seen so far, it may look like the `Series` object is basically interchangeable with a one-dimensional NumPy array. The essential difference is the presence of the index: while the NumPy array has an <i>implicitly defined</i> integer index used to access the values, the Pandas `Series` has an <i>explicitly defined</i> index associated with the values.

The explicit index definition gives the `Series` object additional capabilities. For example, the index need not be an integer, but can consist of values of any desired type. Example of using strings as an index:

In [11]:
data = pd.Series([0.25, 0.5, 0.75, 1.0], index=['a', 'b', 'c', 'd'])
data

a    0.25
b    0.50
c    0.75
d    1.00
dtype: float64

In [12]:
data['b']

0.5

We can even use noncontiguous or nonsequential indices:

In [17]:
data = pd.Series([0.25, 0.5, 0.75, 1.0], index=[2, 5, 3, 7])
data

2    0.25
5    0.50
3    0.75
7    1.00
dtype: float64

In [18]:
data[5]

0.5

<b>Series as a specialized dictionary</b>

You can think of a Pandas `Series` a bit like a specialization of a Python dictionary. A dictionary is a structure that maps arbitrary keys to a set of arbitrary values, and a `Series` is a structure that maps typed keys to a set of typed values. <b>This typing is very important: just as the type-specific compiled code behind a NumPy array makes it more efficient than a Python list for certain operations, the type information of a Pandas `Series` makes it much more efficient than Python dictionaries for certain operations.</b>

We can make the Series-as-dictionary analogy even more clear by constructing a `Series` object directly from a Python dictionary:

In [21]:
population_dict = {'California': 38332521,
                   'Texas': 26448193,
                   'New York': 19651127,
                   'Florida': 19552860,
                   'Illinois': 12882135
                  }

population = pd.Series(population_dict)
population

California    38332521
Texas         26448193
New York      19651127
Florida       19552860
Illinois      12882135
dtype: int64

By default, a `Series` will be created where the index is drawn from the sorted keys. From here, a typical dictionary-style item access can be performed:

In [22]:
population['California']

38332521

Unlike a dictionary, however, the `Series` also supports array-style operations such as slicing:

In [23]:
population['California':'Illinois']

California    38332521
Texas         26448193
New York      19651127
Florida       19552860
Illinois      12882135
dtype: int64

<h3>Constructing Series Objects</h3>

Pandas `Series` creations are all some version of the following:
```python
>>> pd.Series(data, index=index)
```
where index is an optional argument, and data can be one of many entities.

For example, data can be a list or NumPy array, in which case index defaults to an integer sequence:

In [24]:
pd.Series([2, 4, 6])

0    2
1    4
2    6
dtype: int64

* data can be a scalar, which is repeated to fill the specified index:

In [25]:
pd.Series(5, index=[100, 200, 300])

100    5
200    5
300    5
dtype: int64

* data can be a dictionary, in which index defaults to the sorted dictionary keys:

In [26]:
pd.Series({2: 'a', 1: 'b', 3: 'c'})

2    a
1    b
3    c
dtype: object

* Note how in this case, the `Series` is only populated with the explicitly identified keys:

In [28]:
pd.Series({2: 'a', 1: 'b', 3: 'c'}, index=[3, 2])

3    c
2    a
dtype: object

<h3>The Pandas DataFrame Object</h3>

The next fundamental structure in Pandas is the `DataFrame`. Like the `Series` object discussed in the previous section, the `DataFrame` can be thought of either as a generalization of a NumPy array, or as a specialization of a Python dictionary. We'll now take a look at each of these perspectives.

<h4>DataFrame as a generalized NumPy array</h4>

If a `Series` is an analog of a one-dimensional array with flexible indices, a `DataFrame` is an analog of a two-dimensional array with both flexible row indices and flexible column names. Just as you might think of a two-dimensional array as an ordered sequence of aligned one-dimensional columns, <b>you can think of a `DataFrame` as a sequence of aligned `Series` objects. Here, by "aligned", we mean that they share the same index.</b>

To demonstrate this, let's first construct a new `Series` listing the area of each of the five states discussed in the previous section:

In [33]:
area_dict = {'California': 423967, 'Texas': 695662, 'New York': 141297,
             'Florida': 170312, 'Illinois': 149995}
area = pd.Series(area_dict)
area

California    423967
Texas         695662
New York      141297
Florida       170312
Illinois      149995
dtype: int64

* Now that we have this along with the `population Series` from before, we can use a dictionary to construct a single two-dimensional object containing this information:

In [34]:
states = pd.DataFrame({'population': population, 'area': area})
states

,population,area
California,38332521,423967
Texas,26448193,695662
New York,19651127,141297
Florida,19552860,170312
Illinois,12882135,149995


* Like the `Series` object, the `DataFrame` has an index attribute that gives access to the index labels. It also has a columns attribute:

In [35]:
states.index

Index(['California', 'Texas', 'New York', 'Florida', 'Illinois'], dtype='object')

In [36]:
states.columns

Index(['population', 'area'], dtype='object')

Thus the `DataFrame` can be thought of as a generalization of a two-dimensional NumPy array, where both the rows and columns have a generalized index for accessing the data.

<h3>DataFrame as a specialized dictionary</h3>

We can also think of a `DataFrame` as a specialization of a dictionary. Where a dictionary maps a key to a value, a `DataFrame` maps a column name to a `Series` of column data. For example, asking for the 'area' attribute returns the `Series` object containing the areas we saw earlier:

In [37]:
states['area']

California    423967
Texas         695662
New York      141297
Florida       170312
Illinois      149995
Name: area, dtype: int64

In [38]:
type(states['area'])

pandas.core.series.Series

Note the potential confusion here: in a two-dimensional NumPy array, `data[0]` will return the first <i>row</i>. For a `DataFrame`, `data['col0']` will return the first <i>column</i>. Because of this, it is probably better to think about `DataFrames` as generalized dictionaries rather than generalized arrays, though both ways of looking at the situation can be useful. We'll explore more flexible means of indexing `DataFrames` in a bit.

<h4>Constructing DataFrame objects</h4>

A Pandas `DataFrame` object can be constructed in a variety of ways. Here are several examples:

* <b>From a single `Series` object.</b> A `DataFrame` is a collection of `Series` objects, and a single-column `DataFrame` can be constructed from a single `Series`:

In [39]:
pd.DataFrame(population, columns=['population'])

,population
California,38332521
Texas,26448193
New York,19651127
Florida,19552860
Illinois,12882135


* <b>From a list of dicts</b>: Any list of dictionaries can be made into a `DataFrame`. We'll use a simple list comprehension to create some data:

In [40]:
data = [{'a': i, 'b': 2 * i} for i in range(3)]
data

[{'a': 0, 'b': 0}, {'a': 1, 'b': 2}, {'a': 2, 'b': 4}]

In [41]:
pd.DataFrame(data)

,a,b
0,0,0
1,1,2
2,2,4


* Even if some keys in the dictionary are missing, Pandas will fill them with `NaN` values:

In [42]:
pd.DataFrame([{'a': 1, 'b': 2}, {'b': 3, 'c': 4}])

,a,b,c
0,1.0,2,NaN
1,NaN,3,4.0


* <b>From a dictionary of Series objects</b>: A `DataFrame` can be constructed from a dictionary of `Series` objects as well:

In [43]:
pd.DataFrame({'population': population,
              'area': area})

,population,area
California,38332521,423967
Texas,26448193,695662
New York,19651127,141297
Florida,19552860,170312
Illinois,12882135,149995


* <b>From a two-dimensional NumPy array</b>: Given a two-dimensional array of data, we can create a `DataFrame` with any specified column and index names. If omitted, an integer index will be used for each:

In [44]:
pd.DataFrame(np.random.rand(3, 2),
             columns=['foo', 'bar'],
             index=['a', 'b', 'c']
            )

,foo,bar
a,0.165192,0.734091
b,0.633021,0.490757
c,0.316198,0.609857


* <b>From a NumPy structured array</b>: A Pandas `DataFrame` operates much like a structured array, and can be created directly from one:

In [46]:
A = np.zeros(3, dtype=[('A', 'i8'), ('B', 'f8')])
A

array([(0, 0.), (0, 0.), (0, 0.)], dtype=[('A', '<i8'), ('B', '<f8')])

In [47]:
pd.DataFrame(A)

,A,B
0,0,0.0
1,0,0.0
2,0,0.0


<h4>The Pandas Index Object</h4>

We have seen that both the `Series` and `DataFrame` objects contain an explicit <i>index</i> that lets you reference and modify data. This `Index` object is an interesting structure in itself, and can be thought of either as an <i>immutable array</i> or as an <i>ordered set</i> (technically a multiset, as `Index` objects may contain repeated values). Let's construct an `Index` from a list of integers to see its operations:

In [48]:
ind = pd.Index([2, 3, 5, 7, 11])
ind

Int64Index([2, 3, 5, 7, 11], dtype='int64')

* <b>Index as immutable array</b>:

The `Index` object in many ways operates like an array. For example, we can use standard Python indexing notation to retrieve values or slices:

In [49]:
ind[1]

3

In [50]:
ind[::2]

Int64Index([2, 5, 11], dtype='int64')

* Index objects also have many of the attributes familiar from NumPy arrays:

In [51]:
print(ind.size, ind.shape, ind.ndim, ind.dtype)

5 (5,) 1 int64


* One key difference between `Index` objects and NumPy arrays is that indices are immutable -- that is, they cannot be modified via the normal means. This immutability makes it safer to share indices between multiple `DataFrames` and arrays, without the potential for side effects from inadvertent index modification:

In [52]:
ind[1] = 0

TypeError: Index does not support mutable operations

* <b>Index as ordered set</b>:

Pandas objects are designed to facilitate operations such as joins across datasets, which depend on many aspects of set arithmetic. The `Index` object follows many of the conventions used by Python's built-in `set` data structure, so that unions, intersections, differences, and other combinations can be computed in a familiar way:

In [53]:
indA = pd.Index([1, 3, 5, 7, 9])
indB = pd.Index([2, 3, 5, 7, 11])

In [54]:
indA & indB # intersection

<ipython-input-54-f400ec9c4e08>:1: FutureWarning: Index.__and__ operating as a set operation is deprecated, in the future this will be a logical operation matching Series.__and__.  Use index.intersection(other) instead
  indA & indB # intersection


Int64Index([3, 5, 7], dtype='int64')

In [56]:
indA | indB # union

<ipython-input-56-0a17b8447828>:1: FutureWarning: Index.__or__ operating as a set operation is deprecated, in the future this will be a logical operation matching Series.__or__.  Use index.union(other) instead
  indA | indB # union


Int64Index([1, 2, 3, 5, 7, 9, 11], dtype='int64')

In [57]:
indA ^ indB # symmetric difference

<ipython-input-57-aebc1839486a>:1: FutureWarning: Index.__xor__ operating as a set operation is deprecated, in the future this will be a logical operation matching Series.__xor__.  Use index.symmetric_difference(other) instead
  indA ^ indB # symmetric difference


Int64Index([1, 2, 9, 11], dtype='int64')

* These operations can also be accessed via object methods -- e.g.,

```python
indA.intersection(indB)
```

which is also the recommendation in those warning messages.

<h3>Data Indexing and Selection</h3>

* Indexing, slicing, masking, fancy indexing, etc. can be used for NumPy arrays. Here we'll explore similar means of accessing and modifying values in Pandas `Series` and `DataFrame` objects. Let's start with one-dimensional `Series` object, and then move on to the more complicated two-dimensional `DataFrame` object.

<h4>Data Selection in Series</h4>

* As we saw earlier, a `Series` object acts in many ways like a one-dimensional NumPy array, and in many ways like a standard Pythonic dictionary. Keep these two in mind when understanding the patterns of data indexing and selection in these arrays.

* <b>Series as dictionary</b>: mapping keys to collection of values

In [59]:
data = pd.Series([0.25, 0.5, 0.75, 1.0], index=['a', 'b', 'c', 'd'])
data

a    0.25
b    0.50
c    0.75
d    1.00
dtype: float64

In [60]:
'a' in data

True

In [61]:
data.keys()

Index(['a', 'b', 'c', 'd'], dtype='object')

In [62]:
list(data.items())

[('a', 0.25), ('b', 0.5), ('c', 0.75), ('d', 1.0)]

In [63]:
data['e'] = 1.25
data

a    0.25
b    0.50
c    0.75
d    1.00
e    1.25
dtype: float64

* <b>Series as one-dimensional array</b>:

A series builds on this dictionary-like interface and provides array-style item selection via the same basic mechanisms as NumPy arrays -- that is, <i>slices, masking, and fancy indexing</i>. Examples:

In [66]:
# slicing by explicit index
data['a':'c']

a    0.25
b    0.50
c    0.75
dtype: float64

In [67]:
# slicing by implicit integer index
data[0:2]

a    0.25
b    0.50
dtype: float64

In [68]:
# masking
data[(data > 0.3) & (data < 0.8)]

b    0.50
c    0.75
dtype: float64

In [69]:
# fancy indexing
data[['a', 'e']]

a    0.25
e    1.25
dtype: float64

* Among these, slicing may be the most confusing -- notice that when you slice with explicit index, the final index is <i>included</i> in the slice, while when you slice with the implicit integer index, the final index is <i>excluded</i> from the slice.

<h4>Indexers: loc, iloc, and ix</h4>

* These slicing and indexing conventions can be a source of confusion. For example, if your `Series` has an explicit integer index, an indexing operation such as `data[1]` will use the explicit indices, while a slicing operation like `data[1:3]` will use the implicit Python-style index. Example

In [110]:
data = pd.Series(['a', 'b', 'c'], index=[1, 3, 5])
data

1    a
3    b
5    c
dtype: object

In [111]:
# explicit index when indexing
data[1]

'a'

In [112]:
# implicit index when slicing
data[1:3]

3    b
5    c
dtype: object

* Due to the potential confusion in the case of integer indexes, Pandas provides some special <i>indexer</i> attributes that explicitly expose certain indexing schemes. These are not functional methods, but attributes that expose a particular slicing interface to the data in the `Series`.

* First, the `loc` attribute allows indexing and slicing that always references the explicit index:

In [74]:
data.loc[3]

'b'

In [76]:
data.loc[1:4]

1    a
3    b
dtype: object

* The `iloc` attribute allows indexing and slicing that always references the implict, Python-style index:

In [114]:
data.iloc[0]

'a'

In [113]:
data.iloc[1]

'b'

In [81]:
data.iloc[1:4]

3    b
5    c
dtype: object

* The `ix` indexing attribute is a hybrid of the two, and will become more apparent in the context of `DataFrame` objects. (It has actually been deprecated)

* <b>One guiding principle of Python is that "explicit is better than implicit". The explicit nature of `loc` and `iloc` make them very useful in maintaining clean and readable code; especially in the case of integer indexes. It is recommended to use both of these to make code easier to read and understand, and to prevent subtle bugs due to the mixed indexing / slicing convention.</b>

<h4>Data Selection in DataFrame</h4>

* <b>DataFrame as dictionary</b>:

In [115]:
area = pd.Series({'California': 423967, 'Texas': 695662,
                  'New York': 141297, 'Florida': 170312,
                  'Illinois': 149995
                 }
                )

pop = pd.Series({'California': 38332521, 'Texas': 26448193,
                 'New York': 19651127, 'Florida': 19552860,
                 'Illinois': 12882135
                }
               )
data = pd.DataFrame({'area': area, 'pop': pop})
data

,area,pop
California,423967,38332521
Texas,695662,26448193
New York,141297,19651127
Florida,170312,19552860
Illinois,149995,12882135


* The individual `Series` that made up the columns of the `DataFrame` can be accessed via dictionary-style indexing of the column name:

In [116]:
data['area']

California    423967
Texas         695662
New York      141297
Florida       170312
Illinois      149995
Name: area, dtype: int64

* Equivalently, we can also use attribute-style access with column names that are strings:

In [117]:
data.area

California    423967
Texas         695662
New York      141297
Florida       170312
Illinois      149995
Name: area, dtype: int64

* Note that this attribute-style column access actually accesses the same exact object as the dictionary-style access:

In [118]:
data.area is data['area']

True

* Note that if the column names are not strings, this will not work. Also, if the column names conflict with methods of the `DataFrame`, attribute-style access is not possible. For example, the `DataFrame` has a `pop()` method, so data.pop will point to this rather than the "pop" column:

In [119]:
data.pop is data['pop']

False

* In general, avoid the temptation to try column assignment via attribute (i.e., use `data['pop']` = z rather than `data.pop = z`).

In [120]:
data['density'] = data['pop'] / data['area']
data

,area,pop,density
California,423967,38332521,90.413926
Texas,695662,26448193,38.018740
New York,141297,19651127,139.076746
Florida,170312,19552860,114.806121
Illinois,149995,12882135,85.883763


* <b>DataFrame as two-dimensional array</b>

In [121]:
data.values

array([[4.23967000e+05, 3.83325210e+07, 9.04139261e+01],
       [6.95662000e+05, 2.64481930e+07, 3.80187404e+01],
       [1.41297000e+05, 1.96511270e+07, 1.39076746e+02],
       [1.70312000e+05, 1.95528600e+07, 1.14806121e+02],
       [1.49995000e+05, 1.28821350e+07, 8.58837628e+01]])

In [125]:
# transposte the full df to swap rows and columns
data.T

,California,Texas,New York,Florida,Illinois
area,4.239670e+05,6.956620e+05,1.412970e+05,1.703120e+05,1.499950e+05
pop,3.833252e+07,2.644819e+07,1.965113e+07,1.955286e+07,1.288214e+07
density,9.041393e+01,3.801874e+01,1.390767e+02,1.148061e+02,8.588376e+01


* When it comes to indexing `DataFrame` objects, the dictionary-style indexing of columns precludes our ability to simply treat it as a NumPy array. Passing a single index to an array accesses a row:

In [124]:
data.values[0]

array([4.23967000e+05, 3.83325210e+07, 9.04139261e+01])

In [126]:
# passing a single "index" to a df accesses a column
data['area']

California    423967
Texas         695662
New York      141297
Florida       170312
Illinois      149995
Name: area, dtype: int64

* We can again use the `loc` and `iloc` indexers mentioned earlier. Using the `iloc` indexer, we can index the underlying array as if it is a simply NumPy array (using implict Python-style index), but the `DataFrame` index and column labels are maintained in the result:

In [128]:
# first 3 rows, first 2 columns
data.iloc[:3, :2]

,area,pop
California,423967,38332521
Texas,695662,26448193
New York,141297,19651127


In [131]:
# all rows up to and including 'Illinois', columns up to and including 'pop'
data.loc[:'Illinois', :'pop']

,area,pop
California,423967,38332521
Texas,695662,26448193
New York,141297,19651127
Florida,170312,19552860
Illinois,149995,12882135


In [134]:
data.loc[data.density > 100, ['pop', 'density']]

,pop,density
New York,19651127,139.076746
Florida,19552860,114.806121


In [135]:
data.iloc[0, 2] = 90
data

,area,pop,density
California,423967,38332521,90.000000
Texas,695662,26448193,38.018740
New York,141297,19651127,139.076746
Florida,170312,19552860,114.806121
Illinois,149995,12882135,85.883763


In [136]:
data['Florida':'Illinois']

,area,pop,density
Florida,170312,19552860,114.806121
Illinois,149995,12882135,85.883763


In [139]:
data[3:5]

,area,pop,density
Florida,170312,19552860,114.806121
Illinois,149995,12882135,85.883763


In [140]:
data[data.density > 100]

,area,pop,density
New York,141297,19651127,139.076746
Florida,170312,19552860,114.806121


In [141]:
data[data['density'] > 100]

,area,pop,density
New York,141297,19651127,139.076746
Florida,170312,19552860,114.806121


<h3>Operating on Data in Pandas</h3>

* Pandas inherits much of its functionality from NumPy and the ufuncs on pg. 50. It can perform quick element-wise operations, both with basic arithmetic (addition, subtraction, multiplication, etc.), and more sophisticated functions (trigonometric functions, exponential and logarithmic functions, etc.).

In [146]:
rng = np.random.RandomState(42)
ser = pd.Series(rng.randint(0, 10, 4))
ser

0    6
1    3
2    7
3    4
dtype: int64

In [147]:
df = pd.DataFrame(rng.randint(0, 10, (3, 4)),
                  columns=['A', 'B', 'C', 'D'])
df

,A,B,C,D
0,6,9,2,6
1,7,4,3,7
2,7,2,5,4


In [196]:
np.exp(ser)

0     403.428793
1      20.085537
2    1096.633158
3      54.598150
dtype: float64

In [197]:
np.sin(df * np.pi / 4)

,A,B,C,D
0,-1.000000,7.071068e-01,1.000000,-1.000000e+00
1,-0.707107,1.224647e-16,0.707107,-7.071068e-01
2,-0.707107,1.000000e+00,-0.707107,1.224647e-16


In [198]:
area = pd.Series({'Alaska': 1723337, 'Texas': 695662,
                  'California': 423967}, name='area')
population = pd.Series({'California': 38332521, 'Texas': 26448193,
                        'New York': 19651127}, name='population')

In [199]:
population

California    38332521
Texas         26448193
New York      19651127
Name: population, dtype: int64

In [200]:
area

Alaska        1723337
Texas          695662
California     423967
Name: area, dtype: int64

In [201]:
population / area

Alaska              NaN
California    90.413926
New York            NaN
Texas         38.018740
dtype: float64

* The resulting array contains the <i>union</i> of indices of the two input arrays, which we could determine using standard Python set arithmetic on these indices. Any item for which one or the other does not have an entry is marked with `NaN` since it is missing. 

In [206]:
area.index | population.index

<ipython-input-206-ff558a211efb>:1: FutureWarning: Index.__or__ operating as a set operation is deprecated, in the future this will be a logical operation matching Series.__or__.  Use index.union(other) instead
  area.index | population.index


Index(['Alaska', 'California', 'New York', 'Texas'], dtype='object')

In [207]:
A = pd.Series([2, 4, 6], index=[0, 1, 2])
B = pd.Series([1, 3, 5], index=[1, 2, 3])
A + B

0    NaN
1    5.0
2    9.0
3    NaN
dtype: float64

* If using `NaN` values is not desired behavior, we can modify the fill value using appropriate object methods in place of the operators. Using `A.add(B)` is equivalent to `A + B`, but allows optional explicit specification of the fill value for any elements in `A` or `B` that might be missing:

In [208]:
A.add(B, fill_value=0)

0    2.0
1    5.0
2    9.0
3    5.0
dtype: float64